# 🎙️ Dynode — Voice Clone Notebook (Hindi, XTTS-v2)

**For campaign operators.** Run each cell top-to-bottom (Runtime ▸ Run all, or ▶ on each).
You upload the speaker's video/audio + your Excel name list, and download a ZIP of greeting
clips spoken in that person's cloned voice.

**First:** set the GPU. *Runtime ▸ Change runtime type ▸ Hardware accelerator = T4 GPU ▸ Save.*

### Two quality modes (chosen automatically)
- **Accurate (fine-tune, ~90%)** — needs **5–10+ minutes** of clean single-speaker audio.
- **Fast (zero-shot, ~80%)** — works from a short clip (≥6 s), no training.

If your template video only has a few seconds of the speaker, upload a **longer separate recording**
of the same person to unlock the Accurate mode.


## 1 · Setup (install + GPU check)
_Ignore any red pip warnings; they're normal on Colab. If a later cell fails on import, do Runtime ▸ Restart session and run again from here._


In [ ]:
import os, subprocess, sys
os.environ['COQUI_TOS_AGREED'] = '1'   # accept XTTS model license non-interactively

# GPU check
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('⚠️  No GPU! Set Runtime ▸ Change runtime type ▸ T4 GPU, then Runtime ▸ Restart and rerun.')

!pip -q install coqui-tts faster-whisper pandas openpyxl 2>/dev/null
print('✅ install step done')

## 2 · Upload the speaker reference
Upload the **template video** OR a **longer clean recording** of the same speaker (video or audio is fine).


In [ ]:
from google.colab import files
import os, glob

print('Select the speaker video/audio file...')
up = files.upload()
src_path = list(up.keys())[0]
print('Got:', src_path)

# Extract a clean mono 22.05 kHz wav (strip video, light denoise + loudness)
REF_WAV = 'speaker_ref.wav'
!ffmpeg -y -i "$src_path" -vn -ac 1 -ar 22050 -af "highpass=f=70,afftdn=nf=-25,loudnorm=I=-16:TP=-1.5:LRA=11" "$REF_WAV" -hide_banner -loglevel error

# Report duration
import wave, contextlib
with contextlib.closing(wave.open(REF_WAV,'r')) as w:
    dur = w.getnframes()/float(w.getframerate())
print(f'Reference audio: {REF_WAV}  ({dur:.1f} s)')
if dur < 30:
    print('ℹ️  Short clip → will use FAST (zero-shot) mode. For ~90%, upload 5–10 min of clean speech.')

## 3 · Settings
Edit these if you like, then run. `MODE='auto'` lets the notebook decide fine-tune vs zero-shot by how much audio you gave it.


In [ ]:
LANGUAGE = 'hi'                       # Hindi
GREETING_TEMPLATE = '{} जी नमस्कार'   # {} is replaced by each name. Use '{}' for name-only.
MODE = 'auto'                         # 'auto' | 'fast' | 'accurate'

# Fine-tune controls (only used in Accurate mode). Defaults are sensible for Colab T4.
EPOCHS = 6
MIN_MINUTES_FOR_FINETUNE = 4.0        # below this, fall back to Fast

print('Greeting example →', GREETING_TEMPLATE.format('राहुल'))

## 4 · Decide mode (and prepare data if fine-tuning)
For Accurate mode this transcribes your audio with Whisper and builds a small training set. Takes a few minutes.


In [ ]:
chosen_mode = MODE
train_csv = eval_csv = None

if MODE in ('auto','accurate'):
    print('Analysing audio with Whisper (this builds the fine-tune dataset)...')
    try:
        from TTS.demos.xtts_ft_demo.utils.formatter import format_audio_list
        train_csv, eval_csv, total_secs = format_audio_list(
            [REF_WAV], target_language=LANGUAGE, out_path='dataset/',
        )
        minutes = total_secs/60.0
        print(f'Usable speech found: {minutes:.1f} min')
        if MODE == 'accurate' or minutes >= MIN_MINUTES_FOR_FINETUNE:
            chosen_mode = 'accurate'
        else:
            chosen_mode = 'fast'
            print('Not enough audio for a strong fine-tune → using FAST mode.')
    except Exception as e:
        print('Preprocessing failed, falling back to FAST mode:', e)
        chosen_mode = 'fast'
else:
    chosen_mode = 'fast'

print('▶ MODE =', chosen_mode.upper())

## 5 · Fine-tune (Accurate mode only)
Skipped automatically in Fast mode. On a T4 this is roughly 15–40 min depending on audio length and epochs.

_This is the most version-sensitive cell. If it errors, copy the message — the Fast path in the next cells still works regardless._


In [ ]:
ft_config = ft_vocab = ft_checkpoint = None

if chosen_mode == 'accurate':
    from TTS.demos.xtts_ft_demo.utils.gpt_train import train_gpt
    print('Training... (do not close the tab)')
    out = train_gpt(
        custom_model='', version='main', language=LANGUAGE,
        num_epochs=EPOCHS, batch_size=3, grad_acumm=84,
        train_csv=train_csv, eval_csv=eval_csv, output_path='run/',
    )
    print('train_gpt returned:', out)

    # Be robust to return-tuple differences across versions: locate artifacts on disk.
    import glob, os
    def _find(pattern):
        hits = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime)
        return hits[-1] if hits else None
    ft_config     = _find('run/**/config.json')
    ft_vocab      = _find('run/**/vocab.json')
    ft_checkpoint = _find('run/**/best_model.pth') or _find('run/**/model.pth')
    print('config:', ft_config)
    print('vocab :', ft_vocab)
    print('ckpt  :', ft_checkpoint)
    assert ft_config and ft_vocab and ft_checkpoint, 'Could not locate fine-tuned files; check the log above.'
else:
    print('Fast mode → no training.')

## 6 · Load the voice & make a SAMPLE
Listen to this before generating the whole batch. If it sounds right, continue.


In [ ]:
import torch
synth = None   # synth(text, out_wav) -> writes a wav in the cloned voice

if chosen_mode == 'accurate':
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts import Xtts
    import soundfile as sf, numpy as np
    cfg = XttsConfig(); cfg.load_json(ft_config)
    model = Xtts.init_from_config(cfg)
    model.load_checkpoint(cfg, checkpoint_path=ft_checkpoint, vocab_path=ft_vocab, use_deepspeed=False)
    model.cuda()
    gpt_lat, spk_emb = model.get_conditioning_latents(audio_path=[REF_WAV], gpt_cond_len=30, max_ref_length=60)
    def synth(text, out_wav):
        o = model.inference(text, LANGUAGE, gpt_lat, spk_emb, temperature=0.65, repetition_penalty=5.0, top_p=0.8)
        sf.write(out_wav, np.asarray(o['wav']), 24000)
        return out_wav
else:
    from TTS.api import TTS
    _tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2', gpu=True)
    def synth(text, out_wav):
        _tts.tts_to_file(text=text, speaker_wav=REF_WAV, language=LANGUAGE, file_path=out_wav)
        return out_wav

sample = synth(GREETING_TEMPLATE.format('राहुल'), 'sample.wav')
from IPython.display import Audio, display
print('Sample greeting:')
display(Audio(sample))

## 7 · Batch: speak every name & download ZIP
Upload the **same Excel** your app uses (columns `Name` and `Mobile`). The notebook makes one clip per row in the cloned voice and zips them as `{Name}_{Mobile}.wav` to match your app's folders.


In [ ]:
from google.colab import files
import pandas as pd, os, re, zipfile

print('Select your Excel file (Name + Mobile columns)...')
xl = files.upload()
xlpath = list(xl.keys())[0]
df = pd.read_excel(xlpath, dtype=str).fillna('')
df.columns = [str(c).strip() for c in df.columns]
assert 'Name' in df.columns and 'Mobile' in df.columns, 'Excel must have Name and Mobile columns'
df = df[df['Name'].str.strip() != '']
print(f'{len(df)} names to speak in mode = {chosen_mode.upper()}')

def safe(s):
    return re.sub(r'[^0-9A-Za-z\u0900-\u097F]+', '_', str(s).strip()).strip('_')

out_dir = 'voice_clips'; os.makedirs(out_dir, exist_ok=True)
for i, row in df.iterrows():
    name = str(row['Name']).strip(); mobile = str(row['Mobile']).strip()
    fn = f'{safe(name)}_{safe(mobile)}.wav'
    synth(GREETING_TEMPLATE.format(name), os.path.join(out_dir, fn))
    print(f'[{i+1}/{len(df)}] {fn}')

zip_path = 'cloned_voice_clips.zip'
with zipfile.ZipFile(zip_path, 'w') as z:
    for f in os.listdir(out_dir):
        z.write(os.path.join(out_dir, f), f)
print('✅ Done. Downloading', zip_path)
files.download(zip_path)